# Multi-Tenant RAG with Metadata Filtering in Amazon Bedrock AgentCore

This tutorial demonstrates end-to-end implementation of multi-tenant RAG using:
- Amazon Cognito for authentication with custom tenant attributes
- Amazon Bedrock AgentCore for identity propagation
- Amazon Bedrock Knowledge Bases with metadata filtering
- Strands Agents for intelligent query processing

## Architecture

```
Tenant A → JWT (tenant_id=A) → AgentCore → Strands Agent
                                                  ↓
                                    RetrieveAndGenerate API
                                    + filter: {tenant_id: A}
                                                  ↓
                                    Knowledge Base (pooled)
                                    ├─ Doc1 {tenant_id: A} ✓
                                    ├─ Doc2 {tenant_id: B} ✗
                                    └─ Doc3 {tenant_id: A} ✓
```

## What You'll Build

1. Cognito User Pool with tenant_id custom attribute
2. S3 bucket with tenant-specific documents and metadata
3. Amazon Bedrock Knowledge Base (single, pooled)
4. AgentCore Runtime with JWT authentication
5. Strands Agent with metadata filtering

## Duration: 30-45 minutes

## Step 1: Install Dependencies

In [2]:
!pip install -q boto3 bedrock-agentcore-starter-toolkit strands-agents pyyaml

In [3]:
import boto3
import json
import time
import base64
import yaml
from boto3.session import Session

boto_session = Session()
region = boto_session.region_name
account_id = boto3.client('sts').get_caller_identity()['Account']

# Initialize clients
cognito_client = boto3.client('cognito-idp', region_name=region)
s3_client = boto3.client('s3', region_name=region)
bedrock_agent_client = boto3.client('bedrock-agent', region_name=region)
iam_client = boto3.client('iam')

print(f"Region: {region}")
print(f"Account ID: {account_id}")

Region: us-west-2
Account ID: 269550163595


## Step 2: Setup Amazon Cognito with Tenant Attributes

In [4]:
user_pool_name = f"rag-multitenant-{int(time.time())}"

user_pool_response = cognito_client.create_user_pool(
    PoolName=user_pool_name,
    Policies={'PasswordPolicy': {'MinimumLength': 8}},
    Schema=[
        {'Name': 'email', 'AttributeDataType': 'String', 'Mutable': True, 'Required': True},
        {'Name': 'tenant_id', 'AttributeDataType': 'String', 'Mutable': True}
    ],
    AutoVerifiedAttributes=['email']
)

user_pool_id = user_pool_response['UserPool']['Id']

app_client_response = cognito_client.create_user_pool_client(
    UserPoolId=user_pool_id,
    ClientName=f"{user_pool_name}-client",
    GenerateSecret=False,
    ExplicitAuthFlows=['ALLOW_USER_PASSWORD_AUTH', 'ALLOW_REFRESH_TOKEN_AUTH']
)

client_id = app_client_response['UserPoolClient']['ClientId']
discovery_url = f"https://cognito-idp.{region}.amazonaws.com/{user_pool_id}/.well-known/openid-configuration"

cognito_config = {
    'user_pool_id': user_pool_id,
    'client_id': client_id,
    'discovery_url': discovery_url
}

print(f"✓ User Pool: {user_pool_id}")
print(f"✓ Client ID: {client_id}")

✓ User Pool: us-west-2_mJG7DL9qu
✓ Client ID: 6bv4dkfrfvr6jm8pvtv0h86iak


In [5]:
test_users = [
    {'username': 'broker-a', 'password': 'Pass123!', 'email': 'broker-a@example.com', 'tenant_id': 'tenant-a'},
    {'username': 'broker-b', 'password': 'Pass123!', 'email': 'broker-b@example.com', 'tenant_id': 'tenant-b'}
]

for user in test_users:
    try:
        cognito_client.admin_create_user(
            UserPoolId=user_pool_id,
            Username=user['username'],
            UserAttributes=[
                {'Name': 'email', 'Value': user['email']},
                {'Name': 'email_verified', 'Value': 'true'},
                {'Name': 'custom:tenant_id', 'Value': user['tenant_id']}
            ],
            TemporaryPassword=user['password'],
            MessageAction='SUPPRESS'
        )
        cognito_client.admin_set_user_password(
            UserPoolId=user_pool_id,
            Username=user['username'],
            Password=user['password'],
            Permanent=True
        )
        print(f"  ✓ {user['username']} (Tenant: {user['tenant_id']})")
    except cognito_client.exceptions.UsernameExistsException:
        print(f"  ↻ {user['username']} exists")

  ✓ broker-a (Tenant: tenant-a)
  ✓ broker-b (Tenant: tenant-b)


## Step 3: Create S3 Bucket with Tenant Documents

In [ ]:
bucket_name = f"rag-multitenant-kb-{account_id}"

try:
    if region == 'us-east-1':
        s3_client.create_bucket(Bucket=bucket_name)
    else:
        s3_client.create_bucket(
            Bucket=bucket_name,
            CreateBucketConfiguration={'LocationConstraint': region}
        )
    print(f"✓ Created S3 bucket: {bucket_name}")
except s3_client.exceptions.BucketAlreadyOwnedByYou:
    print(f"↻ Bucket exists: {bucket_name}")

✓ Created S3 bucket: rag-multitenant-kb-269550163595-1761944078


In [7]:
# Sample documents for each tenant
documents = {
    'tenant-a': {
        'policy.txt': 'Tenant A Insurance Policy: Maximum deductible for auto insurance is $1000. Coverage includes collision and comprehensive.',
        'claims.txt': 'Tenant A Claims History: 5 claims filed in 2024. Average claim amount: $2500.'
    },
    'tenant-b': {
        'policy.txt': 'Tenant B Insurance Policy: Maximum deductible for auto insurance is $500. Premium coverage with roadside assistance.',
        'claims.txt': 'Tenant B Claims History: 2 claims filed in 2024. Average claim amount: $1800.'
    }
}

# Upload documents with metadata
for tenant_id, docs in documents.items():
    for filename, content in docs.items():
        # Upload document
        s3_key = f"{tenant_id}/{filename}"
        s3_client.put_object(Bucket=bucket_name, Key=s3_key, Body=content)
        
        # Upload metadata file
        metadata = {
            "metadataAttributes": {
                "tenant_id": tenant_id,
                "document_type": filename.split('.')[0]
            }
        }
        metadata_key = f"{s3_key}.metadata.json"
        s3_client.put_object(
            Bucket=bucket_name,
            Key=metadata_key,
            Body=json.dumps(metadata)
        )
        print(f"  ✓ {s3_key} + metadata")

print(f"\n✓ Uploaded {sum(len(docs) for docs in documents.values())} documents with metadata")

  ✓ tenant-a/policy.txt + metadata
  ✓ tenant-a/claims.txt + metadata
  ✓ tenant-b/policy.txt + metadata
  ✓ tenant-b/claims.txt + metadata

✓ Uploaded 4 documents with metadata


## Step 4: Create Amazon Bedrock Knowledge Base

In [8]:
# Create IAM role for Knowledge Base
kb_role_name = f"BedrockKBRole-{int(time.time())}"

trust_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "bedrock.amazonaws.com"},
        "Action": "sts:AssumeRole"
    }]
}

kb_role_response = iam_client.create_role(
    RoleName=kb_role_name,
    AssumeRolePolicyDocument=json.dumps(trust_policy)
)
kb_role_arn = kb_role_response['Role']['Arn']

# Attach policies
kb_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": ["s3:GetObject", "s3:ListBucket"],
            "Resource": [f"arn:aws:s3:::{bucket_name}/*", f"arn:aws:s3:::{bucket_name}"]
        },
        {
            "Effect": "Allow",
            "Action": "bedrock:InvokeModel",
            "Resource": "*"
        }
    ]
}

iam_client.put_role_policy(
    RoleName=kb_role_name,
    PolicyName='KBPolicy',
    PolicyDocument=json.dumps(kb_policy)
)

print(f"✓ Created KB role: {kb_role_arn}")
print("  Waiting 10s for IAM propagation...")
time.sleep(10)

✓ Created KB role: arn:aws:iam::269550163595:role/BedrockKBRole-1761944079
  Waiting 10s for IAM propagation...


In [9]:
import time
import json
import boto3
from botocore.exceptions import ClientError
from opensearchpy import OpenSearch, RequestsHttpConnection, AWSV4SignerAuth

# Initialize clients
aoss_client = boto3.client('opensearchserverless')
bedrock_agent_client = boto3.client('bedrock-agent')
iam_client = boto3.client('iam')

# Get current AWS account ID and credentials
sts_client = boto3.client('sts')
account_id = sts_client.get_caller_identity()['Account']
session = boto3.Session()
credentials = session.get_credentials()

# Generate unique identifiers
timestamp = int(time.time())
collection_name = f'kb-coll-{timestamp}'
policy_prefix = f'kb-{timestamp}'

# Step 1: Update KB role policy
kb_policy_updated = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": ["s3:GetObject", "s3:ListBucket"],
            "Resource": [
                f"arn:aws:s3:::{bucket_name}/*", 
                f"arn:aws:s3:::{bucket_name}"
            ]
        },
        {
            "Effect": "Allow",
            "Action": "bedrock:InvokeModel",
            "Resource": "*"
        },
        {
            "Effect": "Allow",
            "Action": ["aoss:APIAccessAll"],
            "Resource": f"arn:aws:aoss:{region}:{account_id}:collection/*"
        }
    ]
}

iam_client.put_role_policy(
    RoleName=kb_role_name,
    PolicyName='KBPolicy',
    PolicyDocument=json.dumps(kb_policy_updated)
)
print(f"✓ Updated KB role")

# Step 2: Create encryption security policy
encryption_policy = {
    "Rules": [
        {
            "ResourceType": "collection",
            "Resource": [f"collection/{collection_name}"]
        }
    ],
    "AWSOwnedKey": True
}

try:
    aoss_client.create_security_policy(
        name=f'{policy_prefix}-enc',
        type='encryption',
        policy=json.dumps(encryption_policy)
    )
    print(f"✓ Created encryption policy")
except ClientError as e:
    if e.response['Error']['Code'] == 'ConflictException':
        print(f"✓ Encryption policy exists")
    else:
        raise

# Step 3: Create network security policy
network_policy = [
    {
        "Rules": [
            {
                "ResourceType": "collection",
                "Resource": [f"collection/{collection_name}"]
            }
        ],
        "AllowFromPublic": True
    }
]

try:
    aoss_client.create_security_policy(
        name=f'{policy_prefix}-net',
        type='network',
        policy=json.dumps(network_policy)
    )
    print(f"✓ Created network policy")
except ClientError as e:
    if e.response['Error']['Code'] == 'ConflictException':
        print(f"✓ Network policy exists")
    else:
        raise

# Step 4: Get current user/role for data access policy
caller_identity = sts_client.get_caller_identity()
current_principal = caller_identity['Arn']

# Convert assumed-role ARN to role ARN if needed
if ':assumed-role/' in current_principal:
    # Extract role name from assumed-role ARN
    role_name = current_principal.split('/')[-2]
    current_principal = f"arn:aws:iam::{account_id}:role/{role_name}"

data_access_policy = [
    {
        "Rules": [
            {
                "ResourceType": "collection",
                "Resource": [f"collection/{collection_name}"],
                "Permission": [
                    "aoss:CreateCollectionItems",
                    "aoss:DeleteCollectionItems",
                    "aoss:UpdateCollectionItems",
                    "aoss:DescribeCollectionItems"
                ]
            },
            {
                "ResourceType": "index",
                "Resource": [f"index/{collection_name}/*"],
                "Permission": [
                    "aoss:CreateIndex",
                    "aoss:DeleteIndex",
                    "aoss:UpdateIndex",
                    "aoss:DescribeIndex",
                    "aoss:ReadDocument",
                    "aoss:WriteDocument"
                ]
            }
        ],
        "Principal": [
            kb_role_arn,
            current_principal  # Add current user/role for index creation
        ]
    }
]

try:
    aoss_client.create_access_policy(
        name=f'{policy_prefix}-acc',
        type='data',
        policy=json.dumps(data_access_policy)
    )
    print(f"✓ Created data access policy")
except ClientError as e:
    if e.response['Error']['Code'] == 'ConflictException':
        print(f"✓ Data access policy exists")
    else:
        raise

# Wait for policies to propagate
print("Waiting for policies to propagate...")
time.sleep(15)

# Step 5: Create collection
collection_arn = None
collection_id = None

try:
    collection_response = aoss_client.create_collection(
        name=collection_name,
        type='VECTORSEARCH',
        description='Collection for Bedrock Knowledge Base'
    )
    collection_id = collection_response['createCollectionDetail']['id']
    collection_arn = collection_response['createCollectionDetail']['arn']
    print(f"✓ Created collection: {collection_arn}")
    
except ClientError as e:
    if e.response['Error']['Code'] == 'ConflictException':
        collections = aoss_client.list_collections(
            collectionFilters={'name': collection_name}
        )
        if collections['collectionSummaries']:
            collection_id = collections['collectionSummaries'][0]['id']
            collection_arn = collections['collectionSummaries'][0]['arn']
            print(f"✓ Using existing collection: {collection_arn}")
    else:
        raise

# Step 6: Wait for collection to be active
print("Waiting for collection to be active...")
max_wait = 300
wait_interval = 15
elapsed = 0

while elapsed < max_wait:
    response = aoss_client.batch_get_collection(ids=[collection_id])
    status = response['collectionDetails'][0]['status']
    
    if status == 'ACTIVE':
        collection_endpoint = response['collectionDetails'][0]['collectionEndpoint']
        print(f"✓ Collection is ACTIVE")
        print(f"  Endpoint: {collection_endpoint}")
        break
    elif status == 'FAILED':
        raise Exception("Collection creation failed")
    
    print(f"Status: {status}...")
    time.sleep(wait_interval)
    elapsed += wait_interval

if elapsed >= max_wait:
    raise TimeoutError("Collection did not become active")

# Step 7: Create vector index using opensearchpy
print("Creating vector index...")

# Wait a bit more for data access policy to fully propagate
print("Waiting additional time for data access policy propagation...")
time.sleep(20)

# Get current caller identity for debugging
caller_identity = sts_client.get_caller_identity()
current_arn = caller_identity['Arn']
print(f"Current Principal: {current_arn}")

# Verify we can access the collection
try:
    collection_details = aoss_client.batch_get_collection(ids=[collection_id])
    print(f"✓ Can access collection: {collection_details['collectionDetails'][0]['name']}")
except Exception as e:
    print(f"⚠ Collection access check failed: {e}")

# Set up OpenSearch client with AWS auth
auth = AWSV4SignerAuth(credentials, region, 'aoss')
host = collection_endpoint.replace('https://', '')

os_client = OpenSearch(
    hosts=[{'host': host, 'port': 443}],
    http_auth=auth,
    use_ssl=True,
    verify_certs=True,
    connection_class=RequestsHttpConnection,
    timeout=300,
    pool_maxsize=20
)

# Test connection first
try:
    info = os_client.info()
    print(f"✓ Connected to OpenSearch: {info.get('version', {}).get('number', 'unknown')}")
except Exception as e:
    print(f"⚠ Connection test failed: {e}")
    print("Waiting 30s more for policy propagation...")
    time.sleep(30)
    # Recreate client after waiting
    auth = AWSV4SignerAuth(credentials, region, 'aoss')
    os_client = OpenSearch(
        hosts=[{'host': host, 'port': 443}],
        http_auth=auth,
        use_ssl=True,
        verify_certs=True,
        connection_class=RequestsHttpConnection,
        timeout=300,
        pool_maxsize=20
    )

# Define index with proper mappings for Bedrock KB
index_name = 'bedrock-kb-index'
index_body = {
    "settings": {
        "index.knn": True,
        "number_of_shards": 2,
        "knn.algo_param.ef_search": 512
    },
    "mappings": {
        "properties": {
            "bedrock-knowledge-base-default-vector": {
                "type": "knn_vector",
                "dimension": 1024,  # Titan Embed Text v2
                "method": {
                    "name": "hnsw",
                    "engine": "faiss",
                    "parameters": {
                        "ef_construction": 512,
                        "m": 16
                    }
                }
            },
            "AMAZON_BEDROCK_TEXT_CHUNK": {
                "type": "text"
            },
            "AMAZON_BEDROCK_METADATA": {
                "type": "text",
                "index": False
            }
        }
    }
}

# Create index with retry logic
max_retries = 3
retry_delay = 15

for attempt in range(max_retries):
    try:
        os_client.indices.create(index=index_name, body=index_body)
        print(f"✓ Created vector index: {index_name}")
        break
    except Exception as e:
        error_msg = str(e).lower()
        if 'resource_already_exists_exception' in error_msg:
            print(f"✓ Index already exists: {index_name}")
            break
        elif 'authenticationexception' in error_msg or '401' in error_msg:
            if attempt < max_retries - 1:
                print(f"⚠ Authentication failed (attempt {attempt + 1}/{max_retries})")
                print(f"  Waiting {retry_delay}s for policy propagation...")
                time.sleep(retry_delay)
                # Refresh credentials and recreate client
                credentials = session.get_credentials()
                auth = AWSV4SignerAuth(credentials, region, 'aoss')
                os_client = OpenSearch(
                    hosts=[{'host': host, 'port': 443}],
                    http_auth=auth,
                    use_ssl=True,
                    verify_certs=True,
                    connection_class=RequestsHttpConnection,
                    timeout=300,
                    pool_maxsize=20
                )
            else:
                print(f"✗ Failed after {max_retries} attempts")
                print(f"Error: {e}")
                print("\nTroubleshooting:")
                print(f"1. Check data access policy includes: {current_arn}")
                print(f"2. Verify policy propagation (may take up to 60s)")
                print(f"3. Check OpenSearch Serverless collection status")
                raise
        else:
            print(f"Error creating index: {e}")
            raise

# Wait for index to be ready
print("Waiting for index to be ready...")
time.sleep(10)

# Step 8: Create Knowledge Base with retry logic
kb_name = f"multitenant-kb-{timestamp}"

max_retries = 5
retry_delay = 15

for attempt in range(max_retries):
    try:
        kb_response = bedrock_agent_client.create_knowledge_base(
            name=kb_name,
            roleArn=kb_role_arn,
            knowledgeBaseConfiguration={
                'type': 'VECTOR',
                'vectorKnowledgeBaseConfiguration': {
                    'embeddingModelArn': f"arn:aws:bedrock:{region}::foundation-model/amazon.titan-embed-text-v2:0"
                }
            },
            storageConfiguration={
                'type': 'OPENSEARCH_SERVERLESS',
                'opensearchServerlessConfiguration': {
                    'collectionArn': collection_arn,
                    'vectorIndexName': index_name,
                    'fieldMapping': {
                        'vectorField': 'bedrock-knowledge-base-default-vector',
                        'textField': 'AMAZON_BEDROCK_TEXT_CHUNK',
                        'metadataField': 'AMAZON_BEDROCK_METADATA'
                    }
                }
            }
        )
        
        knowledge_base_id = kb_response['knowledgeBase']['knowledgeBaseId']
        print(f"✓ Created Knowledge Base: {knowledge_base_id}")
        break
        
    except ClientError as e:
        error_msg = str(e)
        if 'no such index' in error_msg.lower():
            if attempt < max_retries - 1:
                print(f"⚠ Index not yet visible to Bedrock (attempt {attempt + 1}/{max_retries})")
                print(f"  Waiting {retry_delay}s for index propagation...")
                time.sleep(retry_delay)
            else:
                print(f"✗ Index still not found after {max_retries} attempts")
                print(f"\nTroubleshooting:")
                print(f"1. Verify index exists in OpenSearch Serverless collection")
                print(f"2. Check KB role has proper AOSS permissions")
                print(f"3. Collection: {collection_endpoint}")
                print(f"4. Index name: {index_name}")
                raise
        else:
            # Different error, re-raise immediately
            raise

# Save configuration
kb_config = {
    'kb_id': knowledge_base_id,
    'kb_arn': kb_response['knowledgeBase']['knowledgeBaseArn'],
    'collection_arn': collection_arn,
    'collection_name': collection_name,
    'collection_endpoint': collection_endpoint,
    'index_name': index_name
}

print(f"\n✓ Configuration:")
print(json.dumps(kb_config, indent=2))

✓ Updated KB role
✓ Created encryption policy
✓ Created network policy
✓ Created data access policy
Waiting for policies to propagate...
✓ Created collection: arn:aws:aoss:us-west-2:269550163595:collection/c5b8q4c3tly1v246atri
Waiting for collection to be active...
Status: CREATING...
Status: CREATING...
✓ Collection is ACTIVE
  Endpoint: https://c5b8q4c3tly1v246atri.us-west-2.aoss.amazonaws.com
Creating vector index...
Waiting additional time for data access policy propagation...
Current Principal: arn:aws:iam::269550163595:user/sanghwa
✓ Can access collection: kb-coll-1761944090
⚠ Connection test failed: NotFoundError(404, '')
Waiting 30s more for policy propagation...
✓ Created vector index: bedrock-kb-index
Waiting for index to be ready...
✓ Created Knowledge Base: EF9OJLTDWB

✓ Configuration:
{
  "kb_id": "EF9OJLTDWB",
  "kb_arn": "arn:aws:bedrock:us-west-2:269550163595:knowledge-base/EF9OJLTDWB",
  "collection_arn": "arn:aws:aoss:us-west-2:269550163595:collection/c5b8q4c3tly1v246

In [11]:
# Create Data Source
ds_response = bedrock_agent_client.create_data_source(
    knowledgeBaseId=knowledge_base_id,
    name=f"s3-datasource-{int(time.time())}",
    dataSourceConfiguration={
        'type': 'S3',
        's3Configuration': {
            'bucketArn': f"arn:aws:s3:::{bucket_name}"
        }
    }
)

data_source_id = ds_response['dataSource']['dataSourceId']
print(f"✓ Created Data Source: {data_source_id}")

✓ Created Data Source: 4OK8Y8LFNP


In [12]:
# Start ingestion job
print("Starting ingestion job...")
ingestion_response = bedrock_agent_client.start_ingestion_job(
    knowledgeBaseId=knowledge_base_id,
    dataSourceId=data_source_id
)

ingestion_job_id = ingestion_response['ingestionJob']['ingestionJobId']

# Wait for completion
while True:
    status_response = bedrock_agent_client.get_ingestion_job(
        knowledgeBaseId=knowledge_base_id,
        dataSourceId=data_source_id,
        ingestionJobId=ingestion_job_id
    )
    status = status_response['ingestionJob']['status']
    print(f"  Status: {status}")
    
    if status in ['COMPLETE', 'FAILED']:
        break
    time.sleep(10)

if status == 'COMPLETE':
    print("✓ Ingestion complete")
else:
    raise Exception(f"Ingestion failed: {status}")

Starting ingestion job...
  Status: STARTING
  Status: COMPLETE
✓ Ingestion complete


## Step 5: Deploy RAG Agent with Metadata Filtering

In [18]:
from bedrock_agentcore_starter_toolkit import Runtime

runtime = Runtime()

runtime.configure(
    entrypoint="rag_metadata_agent.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    region=region,
    agent_name="rag_metadata",
    authorizer_configuration={
        "customJWTAuthorizer": {
            "discoveryUrl": cognito_config['discovery_url'],
            "allowedAudience": [cognito_config['client_id']]
        }
    }
)

# Add Authorization header forwarding
with open('.bedrock_agentcore.yaml', 'r') as f:
    config = yaml.safe_load(f)

config['agents']['rag_metadata']['request_header_configuration'] = {
    'requestHeaderAllowlist': ['Authorization']
}

with open('.bedrock_agentcore.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print("✓ Configured agent")

Entrypoint parsed: file=/Users/sanghwa/workspaces/sketch/agentcore-identity-runtime-propagation/identity3/rag_metadata_agent.py, bedrock_agentcore_name=rag_metadata_agent
Memory configured with STM only
Configuring BedrockAgentCore agent: rag_metadata
Will create new memory with mode: STM_ONLY
Memory configuration: Short-term memory only
Found existing memory ID from previous launch: rag_metadata_mem-x3yyVYFOgK
Generated Dockerfile: Dockerfile
Generated .dockerignore: /Users/sanghwa/workspaces/sketch/agentcore-identity-runtime-propagation/identity3/.dockerignore
Keeping 'rag_metadata' as default agent
Bedrock AgentCore configured: /Users/sanghwa/workspaces/sketch/agentcore-identity-runtime-propagation/identity3/.bedrock_agentcore.yaml


✓ Configured agent


In [19]:
print("Deploying agent...\n")
launch_result = runtime.launch(auto_update_on_conflict=True)
print(f"\n✓ Deployed: {launch_result.agent_arn}")

🚀 CodeBuild mode: building in cloud (RECOMMENDED - DEFAULT)
   • Build ARM64 containers in the cloud with CodeBuild
   • No local Docker required
💡 Available deployment modes:
   • runtime.launch()                           → CodeBuild (current)
   • runtime.launch(local=True)                 → Local development
   • runtime.launch(local_build=True)           → Local build + cloud deploy (NEW)
Creating memory resource for agent: rag_metadata


Deploying agent...



✅ MemoryManager initialized for region: us-west-2
🔎 Retrieving memory resource with ID: rag_metadata_mem-x3yyVYFOgK...
  Found memory: rag_metadata_mem-x3yyVYFOgK
Found existing memory in cloud: rag_metadata_mem-x3yyVYFOgK
Existing memory has 0 strategies
✅ Using existing STM-only memory
Starting CodeBuild ARM64 deployment for agent 'rag_metadata' to account 269550163595 (us-west-2)
Setting up AWS resources (ECR repository, execution roles)...
Getting or creating ECR repository for agent: rag_metadata
✅ ECR repository available: 269550163595.dkr.ecr.us-west-2.amazonaws.com/bedrock-agentcore-rag_metadata
Getting or creating execution role for agent: rag_metadata
Using AWS region: us-west-2, account ID: 269550163595
Role name: AmazonBedrockAgentCoreSDKRuntime-us-west-2-6da692c9d1


✅ Reusing existing ECR repository: 269550163595.dkr.ecr.us-west-2.amazonaws.com/bedrock-agentcore-rag_metadata


✅ Reusing existing execution role: arn:aws:iam::269550163595:role/AmazonBedrockAgentCoreSDKRuntime-us-west-2-6da692c9d1
✅ Execution role available: arn:aws:iam::269550163595:role/AmazonBedrockAgentCoreSDKRuntime-us-west-2-6da692c9d1
Preparing CodeBuild project and uploading source...
Getting or creating CodeBuild execution role for agent: rag_metadata
Role name: AmazonBedrockAgentCoreSDKCodeBuild-us-west-2-6da692c9d1
Reusing existing CodeBuild execution role: arn:aws:iam::269550163595:role/AmazonBedrockAgentCoreSDKCodeBuild-us-west-2-6da692c9d1
Using dockerignore.template with 45 patterns for zip filtering
Uploaded source to S3: rag_metadata/source.zip
Updated CodeBuild project: bedrock-agentcore-rag_metadata-builder
Starting CodeBuild build (this may take several minutes)...
Starting CodeBuild monitoring...
🔄 QUEUED started (total: 0s)
✅ QUEUED completed in 1.1s
🔄 PROVISIONING started (total: 1s)
✅ PROVISIONING completed in 7.4s
🔄 DOWNLOAD_SOURCE started (total: 9s)
✅ DOWNLOAD_SOURCE 


✓ Deployed: arn:aws:bedrock-agentcore:us-west-2:269550163595:runtime/rag_metadata-0wxIpMCZce


In [15]:
print("Waiting for agent...")
status_response = runtime.status()
status = status_response.endpoint['status']

while status not in ['READY', 'CREATE_FAILED']:
    time.sleep(10)
    status_response = runtime.status()
    status = status_response.endpoint['status']

time.sleep(60)

print(f"✓ Agent ready: {status}")

✅ MemoryManager initialized for region: us-west-2


Waiting for agent...


🔎 Retrieving memory resource with ID: rag_metadata_mem-x3yyVYFOgK...
  Found memory: rag_metadata_mem-x3yyVYFOgK
Retrieved Bedrock AgentCore status for: rag_metadata


✓ Agent ready: READY


## Step 6: Test Multi-Tenant RAG with Metadata Filtering

In [16]:
# Wait for Knowledge Base to be ready and documents indexed
print("Checking Knowledge Base readiness...")
max_wait = 120
wait_interval = 10
elapsed = 0

kb_ready = False

while elapsed < max_wait:
    try:
        # Check KB status
        kb_detail = bedrock_agent_client.get_knowledge_base(
            knowledgeBaseId=knowledge_base_id
        )
        kb_status = kb_detail['knowledgeBase']['status']
        
        # Check latest ingestion job
        ds_list = bedrock_agent_client.list_data_sources(
            knowledgeBaseId=knowledge_base_id
        )
        
        if ds_list['dataSourceSummaries']:
            data_source_id = ds_list['dataSourceSummaries'][0]['dataSourceId']
            
            jobs = bedrock_agent_client.list_ingestion_jobs(
                knowledgeBaseId=knowledge_base_id,
                dataSourceId=data_source_id,
                maxResults=1
            )
            
            if jobs['ingestionJobSummaries']:
                latest_job = jobs['ingestionJobSummaries'][0]
                job_status = latest_job['status']
                
                print(f"  KB Status: {kb_status}, Ingestion: {job_status}")
                
                if kb_status == 'ACTIVE' and job_status == 'COMPLETE':
                    # Get ingestion stats
                    job_detail = bedrock_agent_client.get_ingestion_job(
                        knowledgeBaseId=knowledge_base_id,
                        dataSourceId=data_source_id,
                        ingestionJobId=latest_job['ingestionJobId']
                    )
                    
                    stats = job_detail['ingestionJob'].get('statistics', {})
                    docs_indexed = stats.get('numberOfNewDocumentsIndexed', 0)
                    
                    print(f"  Documents indexed: {docs_indexed}")
                    
                    if docs_indexed > 0:
                        print("✓ Knowledge Base is ready with indexed documents")
                        kb_ready = True
                        break
                    else:
                        print("⚠ No documents indexed yet")
                        
                elif job_status == 'FAILED':
                    print("✗ Ingestion job failed")
                    break
            else:
                print("  No ingestion jobs found")
        
        time.sleep(wait_interval)
        elapsed += wait_interval
        
    except Exception as e:
        print(f"  Error checking KB status: {e}")
        time.sleep(wait_interval)
        elapsed += wait_interval

if not kb_ready:
    print("⚠ KB status unclear, but proceeding with test...")

Checking Knowledge Base readiness...
  KB Status: ACTIVE, Ingestion: COMPLETE
  Documents indexed: 4
✓ Knowledge Base is ready with indexed documents


In [20]:
def get_tokens(username, password):
    response = cognito_client.initiate_auth(
        ClientId=cognito_config['client_id'],
        AuthFlow='USER_PASSWORD_AUTH',
        AuthParameters={'USERNAME': username, 'PASSWORD': password}
    )
    return response['AuthenticationResult']

# Test Tenant A
print("TEST: Tenant A - Auto Insurance Deductible")
print("="*60)

broker_a_tokens = get_tokens('broker-a', 'Pass123!')
id_token_a = broker_a_tokens['IdToken']

response_a = runtime.invoke(
    {
        "prompt": "What is the maximum deductible for auto insurance?",
        "knowledge_base_id": knowledge_base_id
    },
    bearer_token=id_token_a
)

resp_data = response_a.get('response')
if isinstance(resp_data, str):
    parsed = json.loads(resp_data)
else:
    parsed = resp_data

print(f"\nResponse: {parsed['response']}")
print(f"\nTenant: {parsed['user_identity']['tenant_id']}")
print(f"RAG Used: {parsed['rag_used']}")
print(f"Citations: {parsed.get('citations_count', 0)}")
print(f"Metadata Filter: {parsed.get('metadata_filter')}")

TEST: Tenant A - Auto Insurance Deductible


✅ MemoryManager initialized for region: us-west-2
Memory is active, proceeding with invoke



Response: The maximum deductible for auto insurance for Tenant A is $1000.

Tenant: tenant-a
RAG Used: True
Citations: 1
Metadata Filter: {'equals': {'key': 'tenant_id', 'value': 'tenant-a'}}


In [21]:
# Test Tenant B
print("\nTEST: Tenant B - Auto Insurance Deductible")
print("="*60)

broker_b_tokens = get_tokens('broker-b', 'Pass123!')
id_token_b = broker_b_tokens['IdToken']

response_b = runtime.invoke(
    {
        "prompt": "What is the maximum deductible for auto insurance?",
        "knowledge_base_id": knowledge_base_id
    },
    bearer_token=id_token_b
)

resp_data = response_b.get('response')
if isinstance(resp_data, str):
    parsed = json.loads(resp_data)
else:
    parsed = resp_data

print(f"\nResponse: {parsed['response']}")
print(f"\nTenant: {parsed['user_identity']['tenant_id']}")
print(f"RAG Used: {parsed['rag_used']}")
print(f"Citations: {parsed.get('citations_count', 0)}")
print(f"Metadata Filter: {parsed.get('metadata_filter')}")

print("\n" + "="*60)
print("✓ Multi-tenant isolation verified!")
print("  Tenant A sees $1000 deductible")
print("  Tenant B sees $500 deductible")
print("  Same query, different results based on tenant_id filter")


TEST: Tenant B - Auto Insurance Deductible

Response: According to the search results, the maximum deductible for Tenant B's auto insurance policy is $500.

Tenant: tenant-b
RAG Used: True
Citations: 1
Metadata Filter: {'equals': {'key': 'tenant_id', 'value': 'tenant-b'}}

✓ Multi-tenant isolation verified!
  Tenant A sees $1000 deductible
  Tenant B sees $500 deductible
  Same query, different results based on tenant_id filter


## Summary

### What You Built

1. **Cognito User Pool** with custom tenant_id attribute
2. **S3 Bucket** with tenant-specific documents and metadata files
3. **Knowledge Base** (single, pooled) with metadata indexing
4. **AgentCore Runtime** with JWT authentication
5. **Strands Agent** with metadata filtering for tenant isolation

### Key Patterns

- **Pool-based multi-tenancy**: Single KB, metadata filtering
- **Metadata files**: `document.txt.metadata.json` format
- **Query-time filtering**: `filter: {equals: {key: tenant_id, value: X}}`
- **Identity propagation**: JWT → AgentCore → Strands → RAG

### Cost Optimization

- Single Knowledge Base for all tenants
- Single OpenSearch Serverless collection
- Metadata filtering at vector DB layer (efficient)

### Security

- JWT validation at AgentCore
- Metadata filter enforced at vector DB
- No cross-tenant data leakage possible

## Next Steps

- Add more metadata fields (region, document_type)
- Implement complex filters (AND, OR conditions)
- Add audit logging for compliance
- Monitor per-tenant usage metrics